<a href="https://colab.research.google.com/github/wohecha/HuggingFace-Unit1/blob/main/notebooks/unit8/unit8_part2-1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%bash
# Install ViZDoom deps from
# https://github.com/mwydmuch/ViZDoom/blob/master/doc/Building.md#-linux

sudo apt install build-essential zlib1g-dev libsdl2-dev libjpeg-dev \
nasm tar libbz2-dev libgtk2.0-dev cmake git libfluidsynth-dev libgme-dev \
libopenal-dev timidity libwildmidi-dev unzip ffmpeg

# Boost libraries
sudo apt install libboost-all-dev

# Lua binding dependencies
sudo apt install liblua5.1-dev

## Download and install Miniconda
wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh

Reading package lists...
Building dependency tree...
Reading state information...
build-essential is already the newest version (12.9ubuntu3).
libbz2-dev is already the newest version (1.0.8-5build1).
libbz2-dev set to manually installed.
libjpeg-dev is already the newest version (8c-2ubuntu10).
libjpeg-dev set to manually installed.
cmake is already the newest version (3.22.1-1ubuntu1.22.04.2).
git is already the newest version (1:2.34.1-1ubuntu1.15).
tar is already the newest version (1.34+dfsg-1ubuntu0.1.22.04.2).
unzip is already the newest version (6.0-26ubuntu3.2).
zlib1g-dev is already the newest version (1:1.2.11.dfsg-2ubuntu9.2).
zlib1g-dev set to manually installed.
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
The following additional packages will be installed:
  autopoint debhelper debugedit dh-autoreconf dh-strip-nondeterminism dwz
  fluid-soundfont-gm freepats gettext gettext-base gir1.2-atk-1.0
  gir1.2-gtk-2.0 gir1.2-ibus-1.0 intltool-debian libao-co



debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 100.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 100.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 


debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is in

In [ ]:
%pwd

'/content'

In [2]:
%ls -lah


total 150M
drwxr-xr-x 1 root root 4.0K Feb 22 16:48 ./
drwxr-xr-x 1 root root 4.0K Feb 22 16:43 ../
drwxr-xr-x 4 root root 4.0K Feb  6 14:31 .config/
-rw-r--r-- 1 root root 150M Dec 16 21:40 Miniconda3-latest-Linux-x86_64.sh
drwxr-xr-x 1 root root 4.0K Feb  6 14:31 sample_data/


In [3]:
%%bash
# It appears there's an issue with the Conda environment and pip paths,
# leading to "ModuleNotFoundError". A full runtime restart is required first.
# After the runtime restarts, please run this cell.

# --- Clean up previous Miniconda installation for a fresh start ---
rm -rf /usr/local/bin/conda /usr/local/condabin /usr/local/etc/profile.d/conda.sh /usr/local/envs /usr/local/lib/python* /usr/local/share/conda

# The previous Miniconda download resulted in 'Miniconda3-latest-Linux-x86_64.sh'.
# Ensure we use the correct file for installation.
chmod +x /content/Miniconda3-latest-Linux-x86_64.sh
sudo /content/Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local -u # -u for unattended installation

# --- Diagnostic commands: Check if conda files exist after installation ---
echo "Contents of /usr/local:"
ls -la /usr/local
echo "\nContents of /usr/local/bin:"
ls -la /usr/local/bin
echo "\nContents of /usr/local/etc/profile.d:"
ls -la /usr/local/etc/profile.d

# Initialize conda for the current shell session and ensure paths are set.
# Use the full path to _conda and explicitly source ~/.bashrc.
/usr/local/_conda init bash
source ~/.bashrc

# Accept Conda terms of service (needed for conda create/install)
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Create the environment, specifying Python 3.12 and the classic solver
conda create --name rl_viz python=3.12 --yes --solver classic

# Install required packages within the rl_viz environment using 'conda run'
# 'conda run' executes a command in a specified environment without activating it in the current shell.
conda run -n rl_viz pip install faster-fifo==1.5.2
conda run -n rl_viz pip install vizdoom
conda run -n rl_viz pip install sample-factory==2.1.1
conda run -n rl_viz pip install sample_factory

# Now execute the Python script using the rl_viz environment's python.
conda run -n rl_viz python -c "
import functools
from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.train import run_rl

from sf_examples.vizdoom.doom.doom_model import make_vizdoom_encoder
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec

def register_vizdoom_envs():
    for env_spec in DOOM_ENVS:
        make_env_func = functools.partial(make_doom_env_from_spec, env_spec)
        register_env(env_spec.name, make_env_func)

def register_vizdoom_models():
    global_model_factory().register_encoder_factory(make_vizdoom_encoder)

def register_vizdoom_components():
    register_vizdoom_envs()
    register_vizdoom_models()

def parse_vizdoom_cfg(argv=None, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser)
    doom_override_defaults(parser)
    final_cfg = parse_full_cfg(parser, argv)
    return final_cfg

register_vizdoom_components()
env = \"doom_health_gathering_supreme\"
cfg = parse_vizdoom_cfg(
    argv=[
        f\"--env={env}\",
        \"--num_workers=8\",
        \"--num_envs_per_worker=4\",
        \"--train_for_env_steps=400\"
        ]
    )
status = run_rl(cfg)
"

PREFIX=/usr/local
Unpacking bootstrapper...
Unpacking payload...

Installing base environment...

Preparing transaction: ...working... done
Executing transaction: ...working... done
installation finished.
Contents of /usr/local:
total 36760
drwxr-xr-x   1 root root     4096 Feb 22 16:49 .
drwxr-xr-x   1 root root     4096 Feb 22 16:43 ..
drwxr-xr-x   1 1000 1000     4096 Feb 22 16:49 bin
drwxr-xr-x   3 root root     4096 Feb 19 14:14 colab
drwxr-xr-x   2 root root     4096 Feb 22 16:49 compiler_compat
-rwxr-xr-x   1 root root 37465192 Feb 22 16:49 _conda
drwxr-xr-x   2 root root     4096 Feb 22 16:49 condabin
drwxr-xr-x   2 root root    12288 Feb 22 16:49 conda-meta
-rw-r--r--   1 root root       24 Feb 22 16:49 .condarc
drwxr-xr-x   2 root root     4096 Feb 22 16:49 condarc.d
lrwxrwxrwx   1 root root       22 Mar 10  2025 cuda -> /etc/alternatives/cuda
lrwxrwxrwx   1 root root       25 Mar 10  2025 cuda-12 -> /etc/alternatives/cuda-12
drwxr-xr-x   1 root root     4096 Mar 10  2025 cud



==> WARNING: A newer version of conda exists. <==
  current version: 25.11.1
  latest version: 26.1.1

Please update conda by running

    $ conda update -n base -c defaults conda

Or to minimize the number of packages updated during conda update use

     conda install conda=26.1.1


[2026-02-22 16:54:13,532][05247] register_encoder_factory: <function make_vizdoom_encoder at 0x7d2f4195a700>
[2026-02-22 16:54:13,538][05247] Saved parameter configuration for experiment default_experiment not found!
[2026-02-22 16:54:13,538][05247] Starting experiment from scratch!
[2026-02-22 16:54:13,549][05247] Experiment dir /content/train_dir/default_experiment already exists!
[2026-02-22 16:54:13,549][05247] Resuming existing experiment from /content/train_dir/default_experiment...
[2026-02-22 16:54:13,549][05247] Weights and Biases integration disabled
[2026-02-22 16:54:18,152][05247] Queried available GPUs: 0

[2026-02-22 16:54:18,152][05247] Environment var CUDA_VISIBLE_DEVICES is 0

[2026-02-

In [4]:
!ls -lah /content/train_dir/default_experiment/

total 36K
drwxr-xr-x 4 root root 4.0K Feb 22 16:54 .
drwxr-xr-x 3 root root 4.0K Feb 22 16:54 ..
drwxr-xr-x 2 root root 4.0K Feb 22 16:54 checkpoint_p0
-rw-r--r-- 1 root root 3.8K Feb 22 16:54 config.json
-rw-r--r-- 1 root root  16K Feb 22 16:54 sf_log.txt
drwxr-xr-x 3 root root 4.0K Feb 22 16:54 .summary


In [ ]:
!cat /content/train_dir/default_experiment/sf_log.txt

[2026-02-22 16:09:23,494][04318] Saving configuration to /content/train_dir/default_experiment/config.json...
[2026-02-22 16:09:23,494][04318] Rollout worker 0 uses device cpu
[2026-02-22 16:09:23,495][04318] Rollout worker 1 uses device cpu
[2026-02-22 16:09:23,495][04318] Rollout worker 2 uses device cpu
[2026-02-22 16:09:23,495][04318] Rollout worker 3 uses device cpu
[2026-02-22 16:09:23,495][04318] Rollout worker 4 uses device cpu
[2026-02-22 16:09:23,495][04318] Rollout worker 5 uses device cpu
[2026-02-22 16:09:23,495][04318] Rollout worker 6 uses device cpu
[2026-02-22 16:09:23,495][04318] Rollout worker 7 uses device cpu
[2026-02-22 16:09:23,575][04318] Using GPUs [0] for process 0 (actually maps to GPUs [0])
[2026-02-22 16:09:23,576][04318] InferenceWorker_p0-w0: min num requests: 2
[2026-02-22 16:09:23,601][04318] Starting all processes...
[2026-02-22 16:09:23,601][04318] Starting process learner_proc0
[2026-02-22 16:09:23,637][04318] Starting all processes...
[2026-02-22 16

In [5]:
%%bash
conda run -n rl_viz python -c "
from sample_factory.enjoy import enjoy
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults

def parse_vizdoom_cfg(argv=None, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser)
    doom_override_defaults(parser)
    final_cfg = parse_full_cfg(parser, argv)
    return final_cfg

env = \"doom_health_gathering_supreme\"
cfg = parse_vizdoom_cfg(
    argv=[
        f\"--env={env}\",
        \"--num_workers=1\",
        \"--save_video\",
        \"--no_render\",
        \"--max_num_episodes=10\"
        ],
    evaluation=True
    )
"

In [ ]:
print(cfg)

In [ ]:
status = enjoy(cfg)

In [6]:
!ls -lah /content/train_dir/default_experiment/

total 36K
drwxr-xr-x 4 root root 4.0K Feb 22 16:54 .
drwxr-xr-x 3 root root 4.0K Feb 22 16:54 ..
drwxr-xr-x 2 root root 4.0K Feb 22 16:54 checkpoint_p0
-rw-r--r-- 1 root root 3.8K Feb 22 16:54 config.json
-rw-r--r-- 1 root root  16K Feb 22 16:54 sf_log.txt
drwxr-xr-x 3 root root 4.0K Feb 22 16:54 .summary


In [7]:
!ls -lah /content/train_dir/default_experiment/

total 36K
drwxr-xr-x 4 root root 4.0K Feb 22 16:54 .
drwxr-xr-x 3 root root 4.0K Feb 22 16:54 ..
drwxr-xr-x 2 root root 4.0K Feb 22 16:54 checkpoint_p0
-rw-r--r-- 1 root root 3.8K Feb 22 16:54 config.json
-rw-r--r-- 1 root root  16K Feb 22 16:54 sf_log.txt
drwxr-xr-x 3 root root 4.0K Feb 22 16:54 .summary


In [ ]:
from base64 import b64encode
from IPython.display import HTML

mp4 = open('/content/train_dir/default_experiment/replay.mp4','rb').read()
data_url = "data:video/mp4;base64," + b64encode(mp4).decode()
HTML("""
<video width=640 controls>
      <source src="%s" type="video/mp4">
</video>
""" % data_url)

FileNotFoundError: [Errno 2] No such file or directory: '/content/train_dir/default_experiment/replay.mp4'

In [11]:
!hf auth login


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 
Add token as git credential? [y/N]: n
Token is valid (permission: fineGrained).
The token `unit3` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `unit3`


In [12]:
%%bash

conda run -n rl_viz python -c "
import functools
from sample_factory.algo.utils.context import global_model_factory
from sample_factory.cfg.arguments import parse_full_cfg, parse_sf_args
from sample_factory.envs.env_utils import register_env
from sample_factory.enjoy import enjoy

from sf_examples.vizdoom.doom.doom_model import make_vizdoom_encoder
from sf_examples.vizdoom.doom.doom_params import add_doom_env_args, doom_override_defaults
from sf_examples.vizdoom.doom.doom_utils import DOOM_ENVS, make_doom_env_from_spec

def register_vizdoom_envs():
    for env_spec in DOOM_ENVS:
        make_env_func = functools.partial(make_doom_env_from_spec, env_spec)
        register_env(env_spec.name, make_env_func)

def register_vizdoom_models():
    global_model_factory().register_encoder_factory(make_vizdoom_encoder)

def register_vizdoom_components():
    register_vizdoom_envs()
    register_vizdoom_models()

def parse_vizdoom_cfg(argv=None, evaluation=False):
    parser, _ = parse_sf_args(argv=argv, evaluation=evaluation)
    add_doom_env_args(parser)
    doom_override_defaults(parser)
    final_cfg = parse_full_cfg(parser, argv)
    return final_cfg

# Register the ViZDoom components before creating the environment
register_vizdoom_components()

env = \"doom_health_gathering_supreme\"
hf_username = \"seb-835\" # insert your HuggingFace username here

cfg = parse_vizdoom_cfg(
    argv=[
        f\"--env={env}\",
        \"--num_workers=1\",
        \"--save_video\",
        \"--no_render\",
        \"--max_num_episodes=10\",
        \"--max_num_frames=100000\",
        \"--push_to_hub\",
        f\"--hf_repository={hf_username}/rl_course_vizdoom_health_gathering_supreme\"
    ],
    evaluation=True
)
status = enjoy(cfg)
"

[2026-02-22 16:59:28,956][10309] register_encoder_factory: <function make_vizdoom_encoder at 0x7a2c2aed6020>
[2026-02-22 16:59:28,963][10309] Loading existing experiment configuration from /content/train_dir/default_experiment/config.json
[2026-02-22 16:59:28,963][10309] Overriding arg 'num_workers' with value 1 passed from command line
[2026-02-22 16:59:28,963][10309] Adding new argument 'no_render'=True that is not in the saved config file!
[2026-02-22 16:59:28,963][10309] Adding new argument 'save_video'=True that is not in the saved config file!
[2026-02-22 16:59:28,963][10309] Adding new argument 'video_frames'=1000000000.0 that is not in the saved config file!
[2026-02-22 16:59:28,963][10309] Adding new argument 'video_name'=None that is not in the saved config file!
[2026-02-22 16:59:28,963][10309] Adding new argument 'max_num_frames'=100000 that is not in the saved config file!
[2026-02-22 16:59:28,964][10309] Adding new argument 'max_num_episodes'=10 that is not in the saved c